In [0]:
display(dbutils.fs.ls("abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/"))

path,name,size,modificationTime
abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/customers.csv,customers.csv,9702,1791232676000
abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/order_items.csv,order_items.csv,62919,1791232676000
abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/orders.csv,orders.csv,38009,1791232676000
abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/payments.csv,payments.csv,28455,1791232676000
abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/products.csv,products.csv,5152,1791232676000
abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/regions.csv,regions.csv,120,1791232676000
abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/stores.csv,stores.csv,777,1791232676000


In [0]:
customers_raw_path = "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/customers.csv"
customers_df = (
    spark.read.format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load(customers_raw_path)
)
display(customers_df)

customer_id,first_name,last_name,email,city,state,country,customer_segment,created_date,updated_date
1,Arjun,Sharma,arjun.sharma1@example.com,Indore,Madhya Pradesh,India,Consumer,2024-10-08,2024-12-09
2,Rahul,Nair,rahul.nair2@example.com,Mumbai,Maharashtra,India,Premium,2025-07-12,2025-08-03
3,Pooja,Sharma,pooja.sharma3@example.com,Pune,Maharashtra,India,Consumer,2024-04-05,2024-05-30
4,Nisha,Khan,nisha.khan4@example.com,Pune,Maharashtra,India,Premium,2025-07-28,2025-09-16
5,Riya,Kulkarni,riya.kulkarni5@example.com,Delhi,Delhi,India,Corporate,2025-04-04,2025-09-01
6,Aarav,Reddy,aarav.reddy6@example.com,Lucknow,Uttar Pradesh,India,Corporate,2024-06-12,2024-12-07
7,Ananya,Joshi,ananya.joshi7@example.com,Nagpur,Maharashtra,India,Consumer,2024-08-08,2024-11-02
8,Aditya,Kulkarni,aditya.kulkarni8@example.com,Mumbai,Maharashtra,India,Premium,2025-01-02,2025-03-31
9,Neha,Reddy,neha.reddy9@example.com,Pune,Maharashtra,India,Consumer,2025-04-15,2025-08-30
10,Sneha,Patil,sneha.patil10@example.com,Kolkata,West Bengal,India,Premium,2024-10-27,2025-04-05


In [0]:
customers_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- updated_date: date (nullable = true)



In [0]:
print("Customer record count:", customers_df.count())

Customer record count: 100


In [0]:
print(customers_df.columns)

['customer_id', 'first_name', 'last_name', 'email', 'city', 'state', 'country', 'customer_segment', 'created_date', 'updated_date']


In [0]:
bronze_customers_path = "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/bronze/customers"

(
    customers_df.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_customers_path)
)

In [0]:
bronze_customers_df = spark.read.format("delta").load(bronze_customers_path)

display(bronze_customers_df)

customer_id,first_name,last_name,email,city,state,country,customer_segment,created_date,updated_date
1,Arjun,Sharma,arjun.sharma1@example.com,Indore,Madhya Pradesh,India,Consumer,2024-10-08,2024-12-09
2,Rahul,Nair,rahul.nair2@example.com,Mumbai,Maharashtra,India,Premium,2025-07-12,2025-08-03
3,Pooja,Sharma,pooja.sharma3@example.com,Pune,Maharashtra,India,Consumer,2024-04-05,2024-05-30
4,Nisha,Khan,nisha.khan4@example.com,Pune,Maharashtra,India,Premium,2025-07-28,2025-09-16
5,Riya,Kulkarni,riya.kulkarni5@example.com,Delhi,Delhi,India,Corporate,2025-04-04,2025-09-01
6,Aarav,Reddy,aarav.reddy6@example.com,Lucknow,Uttar Pradesh,India,Corporate,2024-06-12,2024-12-07
7,Ananya,Joshi,ananya.joshi7@example.com,Nagpur,Maharashtra,India,Consumer,2024-08-08,2024-11-02
8,Aditya,Kulkarni,aditya.kulkarni8@example.com,Mumbai,Maharashtra,India,Premium,2025-01-02,2025-03-31
9,Neha,Reddy,neha.reddy9@example.com,Pune,Maharashtra,India,Consumer,2025-04-15,2025-08-30
10,Sneha,Patil,sneha.patil10@example.com,Kolkata,West Bengal,India,Premium,2024-10-27,2025-04-05


In [0]:
print("Bronze customer count:", bronze_customers_df.count())

Bronze customer count: 100


In [0]:
def load_csv_to_bronze(file_name: str, table_name: str) -> None:
    
    raw_path = (
        "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
        f"raw/{file_name}"
    )
    
    bronze_path = (
        "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
        f"bronze/{table_name}"
    )
    
    print(f"Reading RAW: {raw_path}")
    
    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(raw_path)
    )
    
    print(f"Records read: {df.count()}")
    print("Schema:")
    df.printSchema()
    
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(bronze_path)
    )
    
    print(f"Bronze created: {bronze_path}")

In [0]:
load_csv_to_bronze("products.csv", "products")

Reading RAW: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/products.csv
Records read: 60
Bronze created: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/bronze/products


In [0]:
load_csv_to_bronze("orders.csv", "orders")

Reading RAW: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/orders.csv
Records read: 500
Bronze created: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/bronze/orders


In [0]:
load_csv_to_bronze("order_items.csv", "order_items")

Reading RAW: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/order_items.csv
Records read: 1298
Bronze created: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/bronze/order_items


In [0]:
load_csv_to_bronze("payments.csv", "payments")

Reading RAW: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/payments.csv
Records read: 500
Bronze created: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/bronze/payments


In [0]:
load_csv_to_bronze("stores.csv", "stores")

Reading RAW: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/stores.csv
Records read: 15
Bronze created: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/bronze/stores


In [0]:
load_csv_to_bronze("regions.csv", "regions")

Reading RAW: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/regions.csv
Records read: 6
Bronze created: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/bronze/regions


In [0]:
bronze_config = [
    {"source_file": "customers.csv", "table_name": "customers"},
    {"source_file": "products.csv", "table_name": "products"},
    {"source_file": "orders.csv", "table_name": "orders"},
    {"source_file": "order_items.csv", "table_name": "order_items"},
    {"source_file": "payments.csv", "table_name": "payments"},
    {"source_file": "stores.csv", "table_name": "stores"},
    {"source_file": "regions.csv", "table_name": "regions"}
]

bronze_config

[{'source_file': 'customers.csv', 'table_name': 'customers'},
 {'source_file': 'products.csv', 'table_name': 'products'},
 {'source_file': 'orders.csv', 'table_name': 'orders'},
 {'source_file': 'order_items.csv', 'table_name': 'order_items'},
 {'source_file': 'payments.csv', 'table_name': 'payments'},
 {'source_file': 'stores.csv', 'table_name': 'stores'},
 {'source_file': 'regions.csv', 'table_name': 'regions'}]

In [0]:
bronze_config

[{'source_file': 'customers.csv', 'table_name': 'customers'},
 {'source_file': 'products.csv', 'table_name': 'products'},
 {'source_file': 'orders.csv', 'table_name': 'orders'},
 {'source_file': 'order_items.csv', 'table_name': 'order_items'},
 {'source_file': 'payments.csv', 'table_name': 'payments'},
 {'source_file': 'stores.csv', 'table_name': 'stores'},
 {'source_file': 'regions.csv', 'table_name': 'regions'}]

In [0]:
for config in bronze_config:
    load_csv_to_bronze(
        config["source_file"],
        config["table_name"]
    )

Reading RAW: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/customers.csv
Records read: 100
Schema:
root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- updated_date: date (nullable = true)

Bronze created: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/bronze/customers
Reading RAW: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/raw/products.csv
Records read: 60
Schema:
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- cost_price

In [0]:
for config in bronze_config:
    bronze_path = (
        "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
        f"bronze/{config['table_name']}"
    )
    
    df = spark.read.format("delta").load(bronze_path)
    
    print(
        f"{config['table_name']:15} | "
        f"records = {df.count()}"
    )

customers       | records = 100
products        | records = 60
orders          | records = 500
order_items     | records = 1298
payments        | records = 500
stores          | records = 15
regions         | records = 6
